# SEC EDGAR Pipeline Diagnostic

Step-by-step walkthrough of the EDGAR quarterly fundamentals pipeline for 3 tickers:
- **AAPL** (tech, fiscal year ends Sept) 
- **JPM** (bank, calendar year)
- **WMT** (retail, fiscal year ends Jan)

Each step shows exactly what data comes back from the API and how it gets processed.

In [3]:
import sys
sys.path.append("..")

import requests
import pandas as pd
import numpy as np
import time
import json
from importlib import reload

import src.data_fetch.fetch_fundamentals_quarterly as fq
reload(fq)

SEC_USER_AGENT = fq.SEC_USER_AGENT
print(f"User-Agent: {SEC_USER_AGENT}")

User-Agent: AnthonySacco amsacco97@gmail.com


## Step 1: Ticker → CIK Mapping

SEC EDGAR uses CIK (Central Index Key) numbers, not ticker symbols. We load the mapping from `company_tickers.json`.

In [4]:
TEST_TICKERS = ['AAPL', 'JPM', 'WMT']

cik_map = fq._load_cik_mapping()
print(f"Total tickers in CIK mapping: {len(cik_map):,}")
print()

for ticker in TEST_TICKERS:
    cik = cik_map.get(ticker.upper())
    print(f"{ticker}: CIK = {cik}")
    print(f"  API URL: https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json")

Total tickers in CIK mapping: 10,386

AAPL: CIK = 0000320193
  API URL: https://data.sec.gov/api/xbrl/companyfacts/CIK0000320193.json
JPM: CIK = 0000019617
  API URL: https://data.sec.gov/api/xbrl/companyfacts/CIK0000019617.json
WMT: CIK = 0000104169
  API URL: https://data.sec.gov/api/xbrl/companyfacts/CIK0000104169.json


## Step 2: Fetch Raw Company Facts from EDGAR API

The Company Facts endpoint returns ALL XBRL data ever filed by a company. This includes every financial line item across all 10-Q and 10-K filings.

In [5]:
facts_cache = {}

for ticker in TEST_TICKERS:
    cik = cik_map[ticker.upper()]
    facts = fq._fetch_company_facts(cik)
    facts_cache[ticker] = facts
    
    us_gaap = facts.get('facts', {}).get('us-gaap', {})
    dei = facts.get('facts', {}).get('dei', {})
    
    print(f"\n{'='*60}")
    print(f"{ticker} (CIK: {cik})")
    print(f"  us-gaap tags: {len(us_gaap)}")
    print(f"  dei tags: {len(dei)}")
    print(f"  Company: {facts.get('entityName', 'N/A')}")


AAPL (CIK: 0000320193)
  us-gaap tags: 503
  dei tags: 2
  Company: Apple Inc.

JPM (CIK: 0000019617)
  us-gaap tags: 917
  dei tags: 2
  Company: JPMorgan Chase & Co

WMT (CIK: 0000104169)
  us-gaap tags: 459
  dei tags: 2
  Company: Walmart Inc.


## Step 3: XBRL Tag Discovery

Companies use different XBRL tags for the same concept. For example, revenue might be `Revenues`, `RevenueFromContractWithCustomerExcludingAssessedTax`, or `SalesRevenueNet`. We check which tags each company uses.

In [6]:
from datetime import datetime as dt

key_fields = ['revenue', 'net_income', 'operating_cash_flow', 'total_assets', 'shares_outstanding']

for ticker in TEST_TICKERS:
    facts = facts_cache[ticker]
    us_gaap = facts.get('facts', {}).get('us-gaap', {})
    dei = facts.get('facts', {}).get('dei', {})
    
    print(f"\n{'='*60}")
    print(f"{ticker} - Tag Discovery")
    print(f"{'='*60}")
    
    for field in key_fields:
        tags = fq.XBRL_TAG_MAP.get(field, [])
        print(f"\n  {field}:")
        found_any = False
        for tag in tags:
            for ns_name, ns in [('us-gaap', us_gaap), ('dei', dei)]:
                if tag in ns:
                    tag_obj = ns[tag]
                    for unit_key, entries in tag_obj.get('units', {}).items():
                        filtered = [e for e in entries if e.get('form') in ['10-Q', '10-K', '10-Q/A', '10-K/A']]
                        if filtered:
                            years = sorted(set(e['end'][:4] for e in filtered))
                            # Count single-quarter entries
                            sq = 0
                            for e in filtered:
                                if 'start' in e:
                                    dur = (dt.strptime(e['end'], '%Y-%m-%d') - dt.strptime(e['start'], '%Y-%m-%d')).days
                                    if 60 <= dur <= 120:
                                        sq += 1
                            print(f"    {tag} [{ns_name}/{unit_key}]: {len(filtered)} entries ({sq} single-q), {years[0]}-{years[-1]}")
                            found_any = True
        if not found_any:
            print(f"    (no matching tags found)")


AAPL - Tag Discovery

  revenue:
    Revenues [us-gaap/USD]: 11 entries (8 single-q), 2016-2018
    RevenueFromContractWithCustomerExcludingAssessedTax [us-gaap/USD]: 109 entries (60 single-q), 2017-2025
    SalesRevenueNet [us-gaap/USD]: 188 entries (120 single-q), 2007-2018

  net_income:
    NetIncomeLoss [us-gaap/USD]: 308 entries (188 single-q), 2007-2025

  operating_cash_flow:
    NetCashProvidedByUsedInOperatingActivities [us-gaap/USD]: 127 entries (28 single-q), 2007-2025
    NetCashProvidedByUsedInOperatingActivitiesContinuingOperations [us-gaap/USD]: 27 entries (6 single-q), 2012-2016

  total_assets:
    Assets [us-gaap/USD]: 138 entries (0 single-q), 2008-2025

  shares_outstanding:
    EntityCommonStockSharesOutstanding [dei/shares]: 68 entries (0 single-q), 2009-2026
    CommonStockSharesOutstanding [us-gaap/shares]: 136 entries (0 single-q), 2008-2025

JPM - Tag Discovery

  revenue:
    Revenues [us-gaap/USD]: 111 entries (36 single-q), 2007-2025

  net_income:
    Ne

## Step 4: Tag Union (the key fix)

Companies switch tags over time (e.g., AAPL switched from `SalesRevenueNet` to `RevenueFromContractWithCustomerExcludingAssessedTax` in ~2018 after ASC 606). We **union all matching tags** and deduplicate by `(end, start)` date to get full coverage.

In [7]:
for ticker in TEST_TICKERS:
    facts = facts_cache[ticker]
    
    print(f"\n{'='*60}")
    print(f"{ticker} - Revenue Tag Union")
    print(f"{'='*60}")
    
    # Extract with union
    rev_tags = fq.XBRL_TAG_MAP['revenue']
    combined = fq._extract_fact_series(
        facts, rev_tags, unit='USD',
        form_filter=['10-Q', '10-K', '10-Q/A', '10-K/A']
    )
    
    if not combined.empty and 'start' in combined.columns:
        combined['duration_days'] = (combined['end'] - combined['start']).dt.days
        single_q = combined[(combined['duration_days'] >= 60) & (combined['duration_days'] <= 120)]
        cumulative = combined[(combined['duration_days'] > 120) & (combined['duration_days'] < 400)]
        annual = combined[combined['duration_days'] >= 300]
        
        print(f"  Total entries after union + dedup: {len(combined)}")
        print(f"  Single-quarter (60-120d): {len(single_q)} entries")
        if not single_q.empty:
            print(f"    Year range: {single_q['end'].dt.year.min()}-{single_q['end'].dt.year.max()}")
        print(f"  Cumulative (120-400d): {len(cumulative)} entries")
        print(f"  Annual (300d+): {len(annual)} entries")
    else:
        print(f"  No revenue data found")


AAPL - Revenue Tag Union
  Total entries after union + dedup: 119
  Single-quarter (60-120d): 65 entries
    Year range: 2008-2025
  Cumulative (120-400d): 54 entries
  Annual (300d+): 19 entries

JPM - Revenue Tag Union
  Total entries after union + dedup: 53
  Single-quarter (60-120d): 20 entries
    Year range: 2008-2014
  Cumulative (120-400d): 33 entries
  Annual (300d+): 19 entries

WMT - Revenue Tag Union
  Total entries after union + dedup: 118
  Single-quarter (60-120d): 64 entries
    Year range: 2008-2025
  Cumulative (120-400d): 54 entries
  Annual (300d+): 18 entries


## Step 5: Duration Extraction (Income Statement)

Income statement items are "duration" values covering a time period. The extraction:
1. **Single-quarter entries** (~90 days) are used directly
2. **Cumulative YTD entries** (180d, 270d) are differenced to get single-quarter values
3. **Q4** is derived from annual - sum(Q1+Q2+Q3)

In [8]:
duration_fields = ['revenue', 'net_income', 'operating_cash_flow', 'capex']

for ticker in TEST_TICKERS:
    facts = facts_cache[ticker]
    
    print(f"\n{'='*60}")
    print(f"{ticker} - Duration Field Extraction")
    print(f"{'='*60}")
    
    for field in duration_fields:
        result = fq._extract_quarterly_duration(facts, field)
        if not result.empty:
            result = result.sort_values('quarter_end')
            print(f"\n  {field}: {len(result)} quarters")
            print(f"    Range: {result['quarter_end'].min().strftime('%Y-%m-%d')} to {result['quarter_end'].max().strftime('%Y-%m-%d')}")
            # Show last 4 quarters
            last4 = result.tail(4)[['quarter_end', 'value', 'filed']].copy()
            last4['quarter_end'] = last4['quarter_end'].dt.strftime('%Y-%m-%d')
            last4['filed'] = last4['filed'].dt.strftime('%Y-%m-%d')
            last4['value'] = last4['value'].apply(lambda x: f"${x/1e9:.2f}B" if abs(x) >= 1e9 else f"${x/1e6:.0f}M")
            print(f"    Last 4 quarters:")
            for _, r in last4.iterrows():
                print(f"      {r['quarter_end']}  {r['value']:>12s}  (filed: {r['filed']})")
        else:
            print(f"\n  {field}: no data")


AAPL - Duration Field Extraction

  revenue: 72 quarters
    Range: 2007-09-29 to 2025-12-27
    Last 4 quarters:
      2025-03-29       $95.36B  (filed: 2025-05-02)
      2025-06-28       $94.04B  (filed: 2025-08-01)
      2025-09-27       $25.13B  (filed: 2025-10-31)
      2025-12-27      $143.76B  (filed: 2026-01-30)

  net_income: 72 quarters
    Range: 2007-09-29 to 2025-12-27
    Last 4 quarters:
      2025-03-29       $24.78B  (filed: 2025-05-02)
      2025-06-28       $23.43B  (filed: 2025-08-01)
      2025-09-27       $18.27B  (filed: 2025-10-31)
      2025-12-27       $42.10B  (filed: 2026-01-30)

  operating_cash_flow: 72 quarters
    Range: 2007-09-29 to 2025-12-27
    Last 4 quarters:
      2025-03-29      $-64.37B  (filed: 2025-05-02)
      2025-06-28       $27.87B  (filed: 2025-08-01)
      2025-09-27       $29.73B  (filed: 2025-10-31)
      2025-12-27       $53.92B  (filed: 2026-01-30)

  capex: 50 quarters
    Range: 2013-09-28 to 2025-12-27
    Last 4 quarters:
     

## Step 6: Instant Extraction (Balance Sheet)

Balance sheet items are "instant" values - a snapshot at a point in time. These are simpler: just take the value at each period-end date, deduplicate by `end` keeping the latest filing.

In [ ]:
instant_fields = ['total_assets', 'total_equity', 'total_cash', 'shares_outstanding']

for ticker in TEST_TICKERS:
    facts = facts_cache[ticker]
    
    print(f"\n{'='*60}")
    print(f"{ticker} - Instant Field Extraction")
    print(f"{'='*60}")
    
    for field in instant_fields:
        result = fq._extract_quarterly_instant(facts, field)
        if not result.empty:
            result = result.sort_values('quarter_end')
            print(f"\n  {field}: {len(result)} data points")
            print(f"    Range: {result['quarter_end'].min().strftime('%Y-%m-%d')} to {result['quarter_end'].max().strftime('%Y-%m-%d')}")
            # Show last 4
            last4 = result.tail(4)[['quarter_end', 'value']].copy()
            last4['quarter_end'] = last4['quarter_end'].dt.strftime('%Y-%m-%d')
            if field == 'shares_outstanding':
                last4['value'] = last4['value'].apply(lambda x: f"{x/1e9:.2f}B shares")
            else:
                last4['value'] = last4['value'].apply(lambda x: f"${x/1e9:.2f}B" if abs(x) >= 1e9 else f"${x/1e6:.0f}M")
            print(f"    Last 4:")
            for _, r in last4.iterrows():
                print(f"      {r['quarter_end']}  {r['value']}")
        else:
            print(f"\n  {field}: no data")

## Step 7: Quarter Snapping (Instant → Duration alignment)

Instant fields (especially `shares_outstanding`) may have dates that don't match quarter-ends (e.g., cover page dates like 2024-10-18 instead of 2024-09-28). We snap these to the nearest master quarter date (established by duration fields) within ±45 days.

In [ ]:
for ticker in TEST_TICKERS:
    facts = facts_cache[ticker]
    
    # Get master quarters from a duration field
    rev = fq._extract_quarterly_duration(facts, 'net_income')
    master_dates = sorted(rev['quarter_end'].tolist()) if not rev.empty else []
    
    # Get shares_outstanding raw dates
    shares = fq._extract_quarterly_instant(facts, 'shares_outstanding')
    
    if not shares.empty and master_dates:
        print(f"\n{ticker} shares_outstanding snapping:")
        sample = shares.tail(6).copy()
        for _, row in sample.iterrows():
            raw_date = row['quarter_end']
            snapped = fq._snap_to_nearest(raw_date, master_dates, max_days=45)
            offset = (raw_date - snapped).days if snapped else None
            print(f"  Raw: {raw_date.strftime('%Y-%m-%d')} → Snapped: {snapped.strftime('%Y-%m-%d') if snapped else 'N/A'} (offset: {offset:+d} days)")

## Step 8: Full Quarterly DataFrame Assembly

This is the final step: `_build_quarterly_dataframe` combines all fields, calculates ratios (profit margin, ROE, etc.), and derives EBITDA.

In [ ]:
for ticker in TEST_TICKERS:
    facts = facts_cache[ticker]
    df = fq._build_quarterly_dataframe(facts)
    
    print(f"\n{'='*60}")
    print(f"{ticker} - Final Quarterly DataFrame")
    print(f"{'='*60}")
    print(f"Shape: {df.shape}")
    print(f"Date range: {df['quarter_end_date'].min().strftime('%Y-%m-%d')} to {df['quarter_end_date'].max().strftime('%Y-%m-%d')}")
    print(f"\nColumn coverage (non-null / total):")
    
    coverage = df.drop(columns=['quarter_end_date', 'report_date']).notna().sum().sort_values(ascending=False)
    for col, count in coverage.items():
        pct = count / len(df) * 100
        bar = '█' * int(pct / 5) + '░' * (20 - int(pct / 5))
        print(f"  {col:30s} {count:3d}/{len(df)}  {bar} {pct:5.1f}%")

## Step 9: End-to-End `fetch_quarterly_fundamentals()`

Run the complete public API function and inspect the output (including growth metrics and report dates).

In [ ]:
results = {}

for ticker in TEST_TICKERS:
    result = fq.fetch_quarterly_fundamentals(ticker)
    results[ticker] = result
    
    print(f"\n{'='*60}")
    print(f"{ticker} - fetch_quarterly_fundamentals() output")
    print(f"{'='*60}")
    print(f"Shape: {result.shape}")
    print(f"Columns: {result.columns.tolist()}")
    print(f"Date range: {result['quarter_end_date'].min()} to {result['quarter_end_date'].max()}")
    print(f"\nLast 4 quarters:")
    display_cols = ['quarter_end_date', 'report_date', 'revenue', 'net_income', 
                    'operating_cash_flow', 'total_assets', 'profit_margin', 'roe']
    display_cols = [c for c in display_cols if c in result.columns]
    print(result[display_cols].tail(4).to_string(index=False))

## Step 10: Report Date Validation

EDGAR provides actual SEC filing dates. Let's verify they make sense: report_date should be ~30-60 days after quarter_end_date for most companies.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, ticker in zip(axes, TEST_TICKERS):
    df = results[ticker].copy()
    df['quarter_end_dt'] = pd.to_datetime(df['quarter_end_date'])
    df['report_dt'] = pd.to_datetime(df['report_date'])
    df['filing_lag_days'] = (df['report_dt'] - df['quarter_end_dt']).dt.days
    
    ax.hist(df['filing_lag_days'].dropna(), bins=30, edgecolor='black', alpha=0.7)
    ax.axvline(45, color='red', linestyle='--', label='45-day SEC deadline')
    ax.set_title(f'{ticker} Filing Lag')
    ax.set_xlabel('Days after quarter end')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)
    
    median_lag = df['filing_lag_days'].median()
    print(f"{ticker}: Median filing lag = {median_lag:.0f} days, Min = {df['filing_lag_days'].min():.0f}, Max = {df['filing_lag_days'].max():.0f}")

plt.tight_layout()
plt.show()

## Step 11: Data Completeness Heatmap

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

core_cols = ['revenue', 'net_income', 'operating_income', 'gross_profit', 'ebitda',
             'operating_cash_flow', 'capex', 'total_assets', 'total_equity',
             'total_debt', 'total_cash', 'current_assets', 'current_liabilities',
             'shares_outstanding', 'profit_margin', 'roe', 'roa',
             'debt_to_equity', 'current_ratio']

for ax, ticker in zip(axes, TEST_TICKERS):
    df = results[ticker].copy()
    available_cols = [c for c in core_cols if c in df.columns]
    
    # Create binary matrix: 1 = has data, 0 = NaN
    matrix = df[available_cols].notna().astype(int).T
    matrix.columns = df['quarter_end_date']
    
    ax.imshow(matrix.values, aspect='auto', cmap='RdYlGn', interpolation='nearest')
    ax.set_yticks(range(len(available_cols)))
    ax.set_yticklabels(available_cols, fontsize=7)
    ax.set_title(f'{ticker} - Data Completeness ({len(df)} quarters)', fontsize=10)
    
    # Show every 8th x label
    step = max(1, len(df) // 10)
    ax.set_xticks(range(0, len(df), step))
    ax.set_xticklabels(df['quarter_end_date'].iloc[::step], rotation=45, fontsize=7)

plt.tight_layout()
plt.show()

## Step 12: Comparison Summary

In [ ]:
summary_rows = []
for ticker in TEST_TICKERS:
    df = results[ticker]
    row = {'ticker': ticker, 'quarters': len(df)}
    row['date_range'] = f"{df['quarter_end_date'].min()} to {df['quarter_end_date'].max()}"
    for col in core_cols:
        if col in df.columns:
            row[f'{col}_pct'] = f"{df[col].notna().sum() / len(df) * 100:.0f}%"
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows).set_index('ticker')
print(summary.to_string())